# Lab 5: Web Scraping, APIs, and Topic Modeling

Today, we're diving into the world of extracting data directly from the web. We'll learn how to programmatically interact with websites (**web scraping**) to gather information, how to use official channels (**APIs**) like Reddit's to get structured data, and finally, how to make sense of large amounts of text using **Topic Modeling**.

The web is full of data, and today you'll learn the techniques to retrieve it.

**Why are these skills important?**

*   **Data Acquisition:** You can't analyze data you don't have! Scraping and APIs are fundamental ways to build datasets for Machine Learning, market analysis, or academic research.
*   **Competitive Intelligence:** Want to know what competitors are doing? Scrape product prices, features, or reviews.
*   **Social Insights:** Analyze discussions on platforms like Reddit to understand public opinion, trends, or identify communities interested in specific topics (like *sarmale* vs. *mici*?).
*   **News Aggregation & Monitoring:** Create your own news feed or track mentions of specific keywords across the web.
*   **Understanding Text Data:** Topic modeling helps us automatically discover the hidden themes or subjects within large collections of text, like customer feedback or forum posts.

Let's get our tools ready and start investigating!

## Part 1: Web Scraping with BeautifulSoup - The Static Web Investigator

First, we need to understand the structure of most web pages: **HTML** (Hypertext Markup Language). It's the skeleton of a webpage, using tags like `<html>`, `<head>`, `<body>`, `<h1>`, `<p>`, `<a>`, `<div>`, `<span>`, etc., to organize content.

To parse this structure and extract information from *static* websites (pages where the content is mostly fixed and doesn't change much without a full reload), we'll use a fantastic Python library called **BeautifulSoup**. It helps us navigate the HTML tree like a pro.

Think of BeautifulSoup as your magnifying glass for HTML. 🔎

In [8]:
# Install necessary libraries quietly
!pip install beautifulsoup4 requests pandas --quiet

print("Libraries installed successfully!")

Libraries installed successfully!



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [9]:
# @title Basic HTML Parsing with BeautifulSoup - Romanian News Example

import requests
from bs4 import BeautifulSoup
import pandas as pd # We'll use pandas later

# Let's try scraping headlines from a popular Romanian news site
# Note: Website structures can change! This might need adjustments in the future.
# Always check the website's terms of service regarding scraping.
url = 'https://www.digi24.ro' # Example: Digi24 front page

try:
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}) # Add a User-Agent header
    response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)

    html_content = response.text

    # Parse the HTML content
    soup = BeautifulSoup(html_content, 'html.parser')

    # Let's look at the raw HTML (optional, can be very long!)
    print(soup.prettify()) # Use prettify for a nicer formatted output

    print(f"Successfully fetched content from {url}")
    print(f"Page title: {soup.title.string}") # Get the page title

except requests.exceptions.RequestException as e:
    print(f"Error fetching URL {url}: {e}")
    soup = None # Ensure soup is None if fetching failed

<!DOCTYPE html>
<html lang="ro">
 <head>
  <!-- BEGIN: "FrontendUiMain\View\Helper\WidgetLayoutLayoutHeadAssets"; -->
  <!-- BEGIN Seo HEAD -->
  <title>
   Digi24 - Stiri - Informația la putere!
  </title>
  <meta content="Digi24 aduce în prim plan știri relevante, imparțiale și prezentate cu acuratețe. Digi24.ro iti ofera cele mai noi ştiri interne, externe, economice si politice." name="description"/>
  <link href="https://www.digi24.ro/" rel="canonical"/>
  <link href="https://m.digi24.ro/" media="only screen and (max-width: 980px)" rel="alternate"/>
  <link href="https://m.digi24.ro/" rel="handheld"/>
  <meta content="Digi24" property="og:site_name"/>
  <!-- END Seo HEAD -->
  <!-- BEGIN Facebook HEAD -->
  <meta content="Digi24 - Stiri - Informația la putere!" property="og:title"/>
  <meta content="Digi24 aduce în prim plan știri relevante, imparțiale și prezentate cu acuratețe. Digi24.ro iti ofera cele mai noi ştiri interne, externe, economice si politice." property="og:descript

Okay, we have the HTML! Now, how do we find the specific pieces of information we want, like news headlines?

This is where browser developer tools come in handy. In most browsers (Chrome, Firefox, Edge), you can right-click on an element (like a headline) and select "Inspect" or "Inspect Element". This will open a panel showing the HTML code for that specific element and its surroundings.

You'll typically look for patterns:
*   Are all headlines inside `<h2>` tags?
*   Do the elements containing headlines have a specific `class` attribute (e.g., `class="article-title"`)?
*   Are they within a larger `<div>` or `<article>` tag that groups related content?

Let's try to identify a pattern for headlines on Digi24 (as of March 2025, this might change!). Often, main headlines are in `<h2>` or `<h3>` tags within article elements. Let's assume we find they often use a specific class like `article-title` or similar within an `<article>` tag. *(Self-correction: Actual inspection needed here if running live. For this example, let's use a plausible structure.)*

In [10]:
# @title Extracting Structured Data - Headlines from Digi24

if soup: # Proceed only if fetching was successful
    headlines_data = []

    # Find all article blocks (adjust selector based on actual inspection)
    # Common patterns: <article>, <div class="teaser">, etc.
    # Let's try finding <article> tags first.
    article_blocks = soup.find_all('article')

    if not article_blocks:
        # If <article> tags don't work, try a common div structure (example)
        article_blocks = soup.find_all('div', class_='article-item') # Adjust class name as needed!

    print(f"Found {len(article_blocks)} potential article blocks.")

    # Iterate through the blocks and extract headlines
    for block in article_blocks:
        # Try finding headline tags (h2, h3) within the block
        headline_tag = block.find(['h2', 'h3', 'h4']) # Find first h2, h3 or h4

        # Sometimes headlines are inside links (<a>) within these header tags
        if headline_tag:
            headline_link = headline_tag.find('a')
            if headline_link and headline_link.text.strip():
                 headline_text = headline_link.text.strip()
                 headline_url = headline_link.get('href', 'No URL found')
                 # Make URL absolute if it's relative
                 if headline_url.startswith('/'):
                     headline_url = 'https://www.digi24.ro' + headline_url
            elif headline_tag.text.strip():
                 headline_text = headline_tag.text.strip()
                 headline_url = block.find('a').get('href', 'No URL found') if block.find('a') else 'No URL found'
                 if headline_url.startswith('/'):
                     headline_url = 'https://www.digi24.ro' + headline_url
            else:
                continue # Skip if no text found

            headlines_data.append({'Headline': headline_text, 'URL': headline_url})


    # Create a Pandas DataFrame for easier viewing
    if headlines_data:
        headlines_df = pd.DataFrame(headlines_data)
        print("\n--- Extracted Headlines ---")
        display(headlines_df.head()) # Display the first few headlines
    else:
        print("\nCould not extract headlines with the current selectors. Website structure might have changed.")

else:
    print("HTML content not available for parsing.")

Found 75 potential article blocks.

--- Extracted Headlines ---


,Headline,URL
0,Video Planul PSD pentru ieșirea din guvernul ...,https://www.digi24.ro/stiri/actualitate/politi...
1,Exclusiv Oana Gheorghiu spune cât s-a schimba...,https://www.digi24.ro/stiri/actualitate/politi...
2,"Reacția genială a lui Chivu, în dialog cu iubi...",https://www.digisport.ro/fotbal/serie-a/reacti...
3,Video Întâlnire PNL – UDMR. Kelemen Hunor: „I...,https://www.digi24.ro/stiri/actualitate/politi...
4,Exclusiv Robert Negoiță nu înțelege de ce i s...,https://www.digi24.ro/stiri/actualitate/politi...


### Useful BeautifulSoup Methods Recap

*   `find(tag, attrs={}, class_='...', **kwargs)`: Returns the *first* matching element. Great for unique items.
*   `find_all(tag, attrs={}, class_='...', limit=None, **kwargs)`: Returns a *list* of all matching elements. Perfect for repeating items like headlines, products, or comments.
*   `.text` or `get_text(separator='', strip=True)`: Extracts the human-readable text content from within a tag or tags. `strip=True` is useful for removing extra whitespace.
*   `.get(attribute_name)`: Gets the value of an attribute (e.g., `link_tag.get('href')` to get the URL from an `<a>` tag).
*   `select(css_selector)`: Uses CSS selectors (like in stylesheets) to find elements. Very powerful! E.g., `soup.select('div.article > h2.title')` finds `<h2>` tags with class `title` inside `<div>` tags with class `article`.

### Exercise 1: Timișoara Weather Forecaster 🌦️

**Goal:** Scrape the monthly weather forecast for Timișoara from a weather website using BeautifulSoup.

**Website:** `https://www.accuweather.com/` (Let's use AccuWeather for this example - it often has a clear monthly view. *Note: Check the URL validity and website structure before running.*)

**Tasks:**

1.  **Fetch and Parse:** Get the HTML content of the April forecast page for Timișoara.
2.  **Extract Daily Data:** For each day listed on the page, extract:
    *   The day of the month (e.g., "1", "15", "31").
    *   The maximum predicted temperature.
    *   The minimum predicted temperature.
    *   The general weather description (e.g., 'Însorit', 'Parțial noros', 'Ploaie' - this might be in text or associated with an icon, perhaps in the `title` or `alt` attribute of an image, or a specific `<span>`). *Focus on finding textual descriptions.*
3.  **Handle Temperatures:** Ensure temperatures are stored in Celsius. AccuWeather usually shows Celsius for the Romanian version, but if your scraper shows Fahrenheit, convert using: C = (F - 32) * 5 / 9. *Be prepared for non-numeric values like '--' if data is missing and handle them (e.g., store as `NaN`).*
4.  **Create DataFrame:** Store the extracted data in a Pandas DataFrame with columns: `"day"`, `"max_temp_c"`, `"min_temp_c"`, `"weather_description"`.
5.  **Analysis:**
    *   Calculate and display the average *maximum* temperature for the days scraped.
    *   Calculate and display the average *minimum* temperature for the days scraped.
    *   Find and display the day(s) with the lowest minimum temperature.
    *   Count and display how many upcoming days (from today onwards) are predicted to have 'Ploaie' (Rain) or similar rain-indicating terms in their description. (You might need the current date for this).

**Hints:**
*   Use "Inspect Element" heavily! Look for repeating elements (like `div` or `a` tags) that contain the daily forecast data. Find a common class or structure.
*   The `select()` method might be useful if daily blocks share a common CSS class pattern.
*   Temperature might be inside specific `span` tags. The description might be in a `span` or associated with an `img` tag's `alt`/`title` attribute.
*   Use `try-except` blocks when converting temperatures to numbers to handle potential errors (like '--').
*   Use `pd.to_numeric` with `errors='coerce'` for robust temperature conversion in the DataFrame.
*   Import the `datetime` module to get the current day for the rain forecast analysis.

In [11]:
%pip install cloudscraper beautifulsoup4 pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import cloudscraper
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np

# fetch HTML

scraper = cloudscraper.create_scraper()
url = "https://www.accuweather.com/en/ro/timisoara/290867/april-weather/290867"
html_content = scraper.get(url).text

# BeautifulSoup

soup = BeautifulSoup(html_content, 'html.parser')
weather_data = []

# Look for days boxes
day_panels = soup.find_all('a', class_='monthly-daypanel')

for panel in day_panels:
    # 1. get day
    day_elem = panel.find('div', class_='date')
    day = day_elem.get_text(strip=True) if day_elem else None

    # 2. get temps
    high_elem = panel.find('div', class_='high')
    low_elem = panel.find('div', class_='low')

    # clean text
    max_temp = high_elem.get_text(strip=True).replace('°', '') if high_elem else None
    min_temp = low_elem.get_text(strip=True).replace('°', '') if low_elem else None

    # 3. Get Description
    img_elem = panel.find('img')
    description = img_elem.get('alt', 'N/A') if img_elem else 'N/A'

    # Add to list
    if day and max_temp and min_temp:
        weather_data.append({
            'day': int(day),
            'max_temp_c': round(((float(max_temp) - 32) * 5 / 9), 2),
            'min_temp_c': round(((float(min_temp) - 32) * 5 / 9), 2),
            'weather_description': description
        })

# add to df

df = pd.DataFrame(weather_data)

# clean data
df = df.drop_duplicates(subset=['day']).reset_index(drop=True)


print(df.head(10))

avg_max = df['max_temp_c'].mean()
avg_min = df['min_temp_c'].mean()
lowest_min_val = df['min_temp_c'].min()
days_lowest_min = df[df['min_temp_c'] == lowest_min_val]['day'].tolist()

rain_keywords = ['Rain', 'Showers', 'Ploaie']
rainy_days_count = df[df['weather_description'].str.contains('|'.join(rain_keywords), case=False)].shape[0]

print('\0')
print(f"1. Max temp: {avg_max:.2f}°C")
print(f"2. Min temp: {avg_min:.2f}°C")
print(f"3. Min temp: ({lowest_min_val}°C) in days: {days_lowest_min}")
print(f"4. Rainy days: {rainy_days_count}")

   day  max_temp_c  min_temp_c weather_description
0   29      -11.67      -15.00                 N/A
1   30      -12.22      -15.56                 N/A
2   31      -13.33      -15.56                 N/A
3    1      -10.00      -16.11                 N/A
4    2       -8.89      -16.11                 N/A
5    3       -8.89      -12.78                 N/A
6    4       -7.78      -15.00                 N/A
7    5       -7.22      -16.67                 N/A
8    6       -6.11      -15.56                 N/A
9    7       -8.33      -16.67                 N/A
 
1. Max temp: -8.48°C
2. Min temp: -15.34°C
3. Min temp: (-18.33°C) in days: [12]
4. Rainy days: 0


## Part 2: Handling Dynamic Websites with Playwright - The Interactive Investigator

BeautifulSoup + Requests are great for static pages. But what about websites where content loads *after* the initial page load (using JavaScript), or sites that require you to click buttons, scroll down ("infinite scroll"), or fill forms?

For these **dynamic websites**, we need a tool that can actually control a web browser programmatically. Enter **Playwright**!

Playwright allows our Python script to:
*   Launch real browsers (Chromium, Firefox, WebKit) - even headlessly (without a visible window).
*   Navigate to pages.
*   Wait for specific elements or events to happen.
*   Click buttons, fill input fields, hover over elements.
*   Execute JavaScript within the page context.
*   Take screenshots.
*   Get the HTML *after* JavaScript has done its magic.

Think of Playwright as giving your script the ability to use a browser just like a human would, but much faster and automatically.

In [13]:
# @title Install Playwright and its browsers
%pip install playwright pandas --quiet
!playwright install --with-deps # Installs browsers (Chromium, Firefox, WebKit) and their dependencies
# The '--with-deps' flag is important on Linux environments like Colab

print("Playwright and browsers installed.")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Failed to install browsers
Error: Invalid installation targets: '#', 'Installs', 'browsers', '(Chromium,', 'Firefox,', 'WebKit)', 'and', 'their', 'dependencies'. Expecting one of: android, bidi-chromium, chrome, chrome-beta, chrome-for-testing, chromium, chromium-headless-shell, chromium-tip-of-tree, chromium-tip-of-tree-headless-shell, ffmpeg, firefox, firefox-beta, msedge, msedge-beta, msedge-dev, webkit, webkit-wsl, winldd
Playwright and browsers installed.


In [14]:
# @title Basic Playwright Functionality (Sync - Windows/Jupyter safe)

from playwright.sync_api import sync_playwright

def run_basic_playwright():
    try:
        print("Launching Playwright...")
        with sync_playwright() as p:
            # Launch Chromium (headless=True runs in background)
            browser = p.chromium.launch(headless=True)
            print("Browser launched.")

            page = browser.new_page()
            print("Navigating to Playwright's website...")
            page.goto("https://playwright.dev/python/", timeout=60000)
            print("Page loaded.")

            # Get the page title
            title = page.title()
            print(f"Page Title: {title}")

            # Get the full HTML content after JS execution
            content = page.content()
            print("\n--- Page Content Snippet ---")
            print(content[:500] + "...")

            browser.close()
            print("Browser closed.")
        print("Playwright stopped.")

    except Exception as e:
        print(f"An error occurred: {e}")

run_basic_playwright()

Launching Playwright...
An error occurred: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.


In [15]:
# @title Playwright Interaction Example: Searching on eMAG (Sync)

from playwright.sync_api import sync_playwright
import pandas as pd

def search_emag(search_term="laptop", max_items=5):
    results = []
    print(f"Starting eMAG search for: '{search_term}'")

    try:
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=True)
            page = browser.new_page()

            # Go to eMAG homepage
            print("Navigating to eMAG...")
            page.goto("https://www.emag.ro/", timeout=90000)
            print("eMAG homepage loaded.")

            # Search
            search_input_selector = 'input#searchboxTrigger'
            print(f"Filling search input: '{search_input_selector}'")
            page.fill(search_input_selector, search_term, timeout=60000)

            print("Pressing Enter...")
            page.press(search_input_selector, 'Enter')

            # Wait for results
            results_container_selector = 'div#card_grid'
            print(f"Waiting for results container: '{results_container_selector}'")
            page.wait_for_selector(results_container_selector, state='visible', timeout=90000)
            print("Search results page loaded.")

            # Collect product cards
            product_card_selector = 'div.card-item.card-standard'
            print(f"Looking for product cards: '{product_card_selector}'")
            product_cards = page.query_selector_all(product_card_selector)
            print(f"Found {len(product_cards)} product cards on the first page.")

            for i, card in enumerate(product_cards):
                if i >= max_items:
                    break

                title = "N/A"
                price = "N/A"
                url = "#"

                title_tag = card.query_selector('a.card-v2-title')
                if title_tag:
                    title = title_tag.inner_text()
                    url = title_tag.get_attribute('href')
                    if url and not url.startswith('http'):
                        url = f"https://www.emag.ro{url}"

                price_tag = card.query_selector('p.product-new-price')
                if price_tag:
                    price_parts = price_tag.inner_text()
                    price = ' '.join(price_parts.split()).replace('\n', ' ').strip() if price_parts else "N/A"

                results.append({'Title': title.strip(), 'Price': price, 'URL': url})
                print(f"  - Scraped: {title.strip()} - {price}")

            browser.close()
            print("Browser closed.")
        print("Playwright stopped.")

    except Exception as e:
        print(f"An error occurred during eMAG search: {e}")

    return results

search_results = search_emag(search_term="procesor AMD", max_items=5)

if search_results:
    emag_df = pd.DataFrame(search_results)
    print("\n--- eMAG Search Results ---")
    display(emag_df)
else:
    print("\nNo results scraped from eMAG.")

Starting eMAG search for: 'procesor AMD'
An error occurred during eMAG search: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.

No results scraped from eMAG.


### Exercise 2: Dynamic Data Challenge (Choose A or B)

Now it's your turn to tackle a dynamic website! Choose **one** of the following exercises.

**Option A: Goodreads Top Books Scraper **

**Goal:** Scrape the "Best Books Ever" list from Goodreads, potentially handling pagination with Playwright. While *sometimes* possible with BeautifulSoup by changing page URLs, Playwright is more robust if JS is involved in loading or navigation.

**Website:** `https://www.goodreads.com/list/show/1.Best_Books_Ever`

**Tasks:**

1.  **Navigate & Scrape Pages:** Use Playwright to navigate through the first 3 pages of the list (each page usually contains 100 books, so this gets you the top 300). You'll need to find the "next page" button/link and simulate clicks, waiting for the next page to load each time.
2.  **Extract Book Data:** For each book on these pages, scrape:
    *   **Title:** The title of the book.
    *   **Author:** The author's name.
    *   **Ranking:** The book's rank on the list (e.g., "1", "150", "299"). This might be implicitly by order or explicitly listed.
    *   **Average Rating:** The average user rating (e.g., "4.35 avg rating"). Extract the number.
    *   **Score:** The score assigned by Goodreads voters (e.g., "score: 3,941,839"). Extract the number.
    *   **Description:** The text description of the plot, extract the entire text.
3.  **Create DataFrame:** Store all the collected information (for ~300 books) in a Pandas DataFrame.
4.  **Analysis:**
    *   Display the book with the highest *average rating* and its description.
    *   Display the book with the highest *score*. Are they the same book?

**Hints:**
*   Identify the CSS selectors for the book container, title, author, rating, score, and the "next page" link/button.
*   Use `page.click()` for pagination and `page.wait_for_load_state('networkidle')` or `page.wait_for_selector()` to ensure the next page's content is loaded before scraping.
*   Use loops to iterate through pages and books.
*   Clean the extracted text (e.g., remove "avg rating", "score:", commas from numbers) before converting to numeric types.

---

**Option B: Real-Time Stock Tracker (Yahoo Finance) **

**Goal:** Use Playwright to get near real-time stock information for major tech companies from Yahoo Finance, which heavily relies on JavaScript for updating prices.

**Website:** `https://finance.yahoo.com/` (You'll navigate to specific ticker pages)

**Tickers:** `AAPL` (Apple), `GOOGL` (Alphabet/Google), `MSFT` (Microsoft), `AMZN` (Amazon), `TSLA` (Tesla)

**Tasks:**

1.  **Navigate & Extract:** For each ticker symbol:
    *   Navigate to its specific Yahoo Finance page (e.g., `https://finance.yahoo.com/quote/AAPL`).
    *   Wait for the main price information to load (it might update dynamically).
    *   Extract the **Current Price**. (Hint: The original hint `fin-streamer[data-test="qsp-price"]` is a good starting point, but *verify* it with Inspect Element, as attributes can change).
    *   Extract the **Market Change (Absolute Value)** (e.g., "+2.91" or "-1.50").
    *   Extract the **Market Change (Percentage)** (e.g., "+1.05%" or "-0.88%").
    *   Record the **Date and Time** of the reading.
2.  **Repeat Readings:** Perform the extraction process 5 times for each ticker, perhaps with a small delay (e.g., 10-15 seconds) between readings for the *same* ticker to potentially capture minor fluctuations (Note: If the market is closed, readings will be identical).
3.  **Create DataFrame:** Store all readings (5 tickers * 5 readings = 25 rows) in a Pandas DataFrame with columns: `"Ticker"`, `"Price ($)"`, `"Change ($)"`, `"Change (%)"`, `"Timestamp"`.
4.  **Analysis:**
    *   Display the final DataFrame.
    *   For each ticker, calculate the difference between the first and last price reading you captured.

**Hints:**
*   Use f-strings to construct the URLs for each ticker: `f"https://finance.yahoo.com/quote/{ticker}"`.
*   Yahoo Finance uses specific attributes (like `data-field`, `data-symbol`, `data-test`) on elements, especially `<fin-streamer>`. Use Inspect Element carefully to find reliable selectors for the price and change values. `page.locator()` is very useful here.
*   Use `page.wait_for_selector()` to ensure the elements you need are present before trying to extract text.
*   Use `datetime.now()` from the `datetime` module to get the timestamp for each reading.
*   Use `asyncio.sleep()` for delays between readings if needed.
*   Clean the extracted text (remove '+', '%', '$', commas) and convert to numeric types.

In [16]:
!pip install cloudscraper pandas playwright

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [17]:
!playwright install --with-deps
print("Playwright browsers installed.")


Id     Name            PSJobTypeName   State         HasMoreData     Location             Command                  
--     ----            -------------   -----         -----------     --------             -------                  
1      PowerShell.O...                 NotStarted    False                                ...                      


|                                                                                |   0% of 172.8 MiB
|■■■■■■■■                                                                        |  10% of 172.8 MiB
|■■■■■■■■■■■■■■■■                                                                |  20% of 172.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■                                                        |  30% of 172.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                                |  40% of 172.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■                                        |  50% of 172.8 MiB
|■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■

(node:15672) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
(node:22956) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
(node:15472) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
(node:3716) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to

In [19]:
import requests
import pandas as pd
import re
import time
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "https://www.goodreads.com"
LIST_URL = "https://www.goodreads.com/list/show/1.Best_Books_Ever?page={page}"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}

def extract_numeric(pattern, text, cast_type=float, default=None):
    match = re.search(pattern, text or "")
    if not match:
        return default
    value = match.group(1).replace(",", "").strip()
    try:
        return cast_type(value)
    except ValueError:
        return default

def get_book_description(session, book_url):
    try:
        detail_html = session.get(book_url, headers=HEADERS, timeout=30).text
        detail_soup = BeautifulSoup(detail_html, "html.parser")

        # New Goodreads layouts
        desc_blocks = detail_soup.select("div[data-testid='description'] span")
        desc_texts = [d.get_text(" ", strip=True) for d in desc_blocks if d.get_text(strip=True)]
        if desc_texts:
            return max(desc_texts, key=len)

        # Older Goodreads layout fallback
        old_desc = detail_soup.select_one("div#description span[style='display:none']")
        if old_desc and old_desc.get_text(strip=True):
            return old_desc.get_text(" ", strip=True)

        old_desc_short = detail_soup.select_one("div#description span")
        if old_desc_short and old_desc_short.get_text(strip=True):
            return old_desc_short.get_text(" ", strip=True)

        return "N/A"
    except Exception:
        return "N/A"

def scrape_goodreads_pages(pages=3, fetch_descriptions=True, pause_between_books=0.25):
    extracted_books = []

    with requests.Session() as session:
        for page_num in range(1, pages + 1):
            page_url = LIST_URL.format(page=page_num)
            print(f"--- Processing page {page_num}: {page_url}")

            response = session.get(page_url, headers=HEADERS, timeout=30)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            rows = soup.select("table.tableList tr[itemtype='http://schema.org/Book']")
            print(f"Found {len(rows)} book rows.")

            for row in rows:
                try:
                    rank_text = row.select_one("td.number").get_text(strip=True)
                    rank = int(rank_text.replace(".", ""))

                    title_tag = row.select_one("a.bookTitle")
                    title = title_tag.get_text(" ", strip=True) if title_tag else "N/A"
                    book_href = title_tag.get("href", "") if title_tag else ""
                    book_url = urljoin(BASE_URL, book_href) if book_href else ""

                    author_tag = row.select_one("a.authorName")
                    author = author_tag.get_text(" ", strip=True) if author_tag else "N/A"

                    minirating_text = row.select_one("span.minirating")
                    minirating_text = minirating_text.get_text(" ", strip=True) if minirating_text else ""
                    avg_rating = extract_numeric(r"([\d.]+)\s+avg rating", minirating_text, float, 0.0)

                    score_tag = row.select_one("a[href*='votes']")
                    score_text = score_tag.get_text(" ", strip=True) if score_tag else ""
                    score = extract_numeric(r"score:\s*([\d,]+)", score_text, int, 0)

                    if fetch_descriptions and book_url:
                        description = get_book_description(session, book_url)
                        time.sleep(pause_between_books)
                    else:
                        description = "N/A"

                    extracted_books.append({
                        "Title": title,
                        "Author": author,
                        "Ranking": rank,
                        "Average Rating": avg_rating,
                        "Score": score,
                        "Description": description
                    })
                except Exception:
                    continue

            # Be polite between page requests
            time.sleep(1)

    return pd.DataFrame(extracted_books)

# ==== EXECUTION ====
# Set fetch_descriptions=False if you want a much faster run.
df_books = scrape_goodreads_pages(pages=3, fetch_descriptions=True, pause_between_books=0.25)

if not df_books.empty:
    max_rating_book = df_books.loc[df_books["Average Rating"].idxmax()]
    max_score_book = df_books.loc[df_books["Score"].idxmax()]

    print("\n--- ANALYSIS ---")
    print(f"Highest AVERAGE RATING: {max_rating_book['Title']} ({max_rating_book['Average Rating']})")
    print(f"Description: {max_rating_book['Description'][:400]}...")
    print(f"Highest SCORE: {max_score_book['Title']} ({max_score_book['Score']:,})")
    print(f"Same book for both metrics? {'Yes' if max_rating_book['Title'] == max_score_book['Title'] else 'No'}")
else:
    print("No books extracted. Goodreads may have blocked the request. Try again later.")

--- Processing page 1: https://www.goodreads.com/list/show/1.Best_Books_Ever?page=1
Found 100 book rows.
--- Processing page 2: https://www.goodreads.com/list/show/1.Best_Books_Ever?page=2
Found 100 book rows.
--- Processing page 3: https://www.goodreads.com/list/show/1.Best_Books_Ever?page=3
Found 100 book rows.

--- ANALYSIS ---
Highest AVERAGE RATING: The Addiction Manifesto (4.73)
Description: 2020 International Book Awards Finalist for Health: Addiction & Recovery 2021 Literary Titan Silver Award "Some people won't believe in you, and that's ok, this journey isn't about them. It's about you." The Addiction Manifesto has been uniquely designed to provide you with a new perspective on recovery and will show you that anything is possible. In this deeply personal book, JR Weaver has crafte...
Highest SCORE: The Hunger Games (The Hunger Games, #1) (0)
Same book for both metrics? No
